# ***DATA ANALYSIS USING PANDAS***

### 1. Data Import and Cleaning

In [1]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import MinMaxScaler

In [2]:
from google.colab import drive
drive.mount('/content/drive')
path='/content/drive/MyDrive/Data_Science/data/Retail_Transactions_Practice_Dataset.csv'
path2='/content/drive/MyDrive/Data_Science/data/Product_Catalog_Matching_Table.csv'
df1=pd.read_csv(path)
df2=pd.read_csv(path2)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
df1.head()

,TransactionID,Date,CustomerID,ProductID,Category,Region,Quantity,UnitPrice,DiscountPct,NetSales,PaymentMethod,CustomerRating,Returned,DeliveryDays
0,T00001,2024-11-23,1062,214,Grocery,North,2,17.38,10.0,31.28,Card,5.0,No,1.0
1,T00002,2024-01-16,1154,201,Sports,South,2,113.23,20.0,181.17,Wallet,2.0,No,5.0
2,T00003,2024-01-04,1071,209,Grocery,East,3,350.58,10.0,946.57,Card,1.0,No,2.0
3,T00004,2024-07-02,1186,229,Sports,North,6,303.84,0.0,1823.04,Wallet,1.0,No,7.0
4,T00005,2024-10-22,1169,214,Clothing,North,4,353.76,0.0,1415.04,UPI,1.0,No,5.0


In [4]:
df1.shape

(1020, 14)

In [5]:
# Drop rows with missing values
df_clean = df1.dropna()

In [6]:
df1.isnull().sum()

,0
TransactionID,0
Date,0
CustomerID,0
ProductID,0
Category,20
Region,0
Quantity,0
UnitPrice,0
DiscountPct,188
NetSales,188


In [7]:
# Replace missing numerical values with column mean and median
df1['CustomerRating'] = df1['CustomerRating'].fillna(df1['CustomerRating'].mean())
df1['DeliveryDays'] = df1['DeliveryDays'].fillna(df1['DeliveryDays'].median())

In [8]:
# Replace missing categorical values with column mode
df1['PaymentMethod'] = df1['PaymentMethod'].fillna(df1['PaymentMethod'].mode()[0])
df1['Category'] = df1['Category'].fillna(df1['Category'].mode()[0])

In [9]:
df1.head()

,TransactionID,Date,CustomerID,ProductID,Category,Region,Quantity,UnitPrice,DiscountPct,NetSales,PaymentMethod,CustomerRating,Returned,DeliveryDays
0,T00001,2024-11-23,1062,214,Grocery,North,2,17.38,10.0,31.28,Card,5.0,No,1.0
1,T00002,2024-01-16,1154,201,Sports,South,2,113.23,20.0,181.17,Wallet,2.0,No,5.0
2,T00003,2024-01-04,1071,209,Grocery,East,3,350.58,10.0,946.57,Card,1.0,No,2.0
3,T00004,2024-07-02,1186,229,Sports,North,6,303.84,0.0,1823.04,Wallet,1.0,No,7.0
4,T00005,2024-10-22,1169,214,Clothing,North,4,353.76,0.0,1415.04,UPI,1.0,No,5.0


### 2. Data Transformation

In [12]:
# Sum of two columns using NumPy
df1['GrossAmount'] = np.add(df1['Quantity']*df1['UnitPrice'], 0)

In [13]:
# Square root using NumPy
df1['Sqrt_NetSales'] = np.sqrt(df1['NetSales'].clip(lower=0))

In [14]:
# Normalize numerical column using MinMaxScaler
scaler = MinMaxScaler()
df1['NetSales_Normalized'] = scaler.fit_transform(df1[['NetSales']])

In [15]:
df1.head()

,TransactionID,Date,CustomerID,ProductID,Category,Region,Quantity,UnitPrice,DiscountPct,NetSales,PaymentMethod,CustomerRating,Returned,DeliveryDays,GrossAmount,Sqrt_NetSales,NetSales_Normalized
0,T00001,2024-11-23,1062,214,Grocery,North,2,17.38,10.0,31.28,Card,5.0,No,1.0,34.76,5.592853,0.001699
1,T00002,2024-01-16,1154,201,Sports,South,2,113.23,20.0,181.17,Wallet,2.0,No,5.0,226.46,13.459941,0.040581
2,T00003,2024-01-04,1071,209,Grocery,East,3,350.58,10.0,946.57,Card,1.0,No,2.0,1051.74,30.766378,0.239127
3,T00004,2024-07-02,1186,229,Sports,North,6,303.84,0.0,1823.04,Wallet,1.0,No,7.0,1823.04,42.697072,0.466484
4,T00005,2024-10-22,1169,214,Clothing,North,4,353.76,0.0,1415.04,UPI,1.0,No,5.0,1415.04,37.617017,0.360648


### 3. Merging and Joining Datasets

In [16]:
# Merge two tables based on ProductID
df_merged = pd.merge(df1, df2, on='ProductID', how='inner').fillna(0)

In [17]:
# Left join and handle missing data
df_left = pd.merge(df1, df2, on='ProductID', how='left').fillna({'Brand': 'Unknown'})

In [18]:
# Concatenate DataFrames along columns
df_concat = pd.concat([df1[['NetSales']], df2[['Brand']]], axis=1)

### 4. Grouping and Aggregation


In [19]:
# Using Groupby mean and std
grouped_stats = df1.groupby('Category')['NetSales'].agg(['mean', 'std'])

In [20]:
grouped_stats

,mean,std
Category,,
Clothing,1096.917470,974.023816
Electronics,1000.736446,787.399241
Grocery,1026.316205,814.420565
Home,1125.061333,908.313209
Sports,995.619870,854.356758


In [21]:
# Groupby sum + NumPy function (Log transformation)
region_sum = df1.groupby('Region')['NetSales'].sum()
log_region_sum = np.log1p(region_sum)

In [22]:
region_sum

,NetSales
Region,
East,219400.77
North,217813.58
South,199668.00
West,237533.19


In [23]:
# Pivot Table
pivot_tbl = pd.pivot_table(df1, values='NetSales', index='Category', columns='Region', aggfunc=np.sum)

/tmp/ipykernel_8988/2988202141.py:2: FutureWarning: The provided callable <function sum at 0x7a17f458d300> is currently using DataFrameGroupBy.sum. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "sum" instead.
  pivot_tbl = pd.pivot_table(df1, values='NetSales', index='Category', columns='Region', aggfunc=np.sum)


In [24]:
pivot_tbl

Region,East,North,South,West
Category,,,,
Clothing,48208.23,34892.41,38628.79,60358.87
Electronics,40022.34,50682.88,44225.92,31191.11
Grocery,38351.34,43043.66,41320.22,47653.27
Home,66497.49,51308.37,32640.67,52064.51
Sports,26321.37,37886.26,42852.40,46265.43


### 5. Array Operations and Manipulation

In [25]:
# Elemnt wise operation
sales_arr = df1['NetSales'].to_numpy()
sales_doubled = sales_arr * 2
sales_doubled

array([  62.56,  362.34, 1893.14, ..., 1687.96, 1192.32,  638.98])

In [26]:
# Reshape a NumPy array and assign it back to a new DataFrame column
reshaped_arr = np.reshape(df1['Quantity'].values, (-1, 1))
df1['Reshaped_Feature'] = reshaped_arr

In [27]:
# Filter with sales greater than 1000
high_sales_df = df1[df1['NetSales'] > 1000]

In [28]:
high_sales_df.shape

(360, 18)

### 6. Broadcasting and Vectorized Operations


In [29]:
# Broadcast 10% discount and make  new column
df1['DiscountedPrice'] = df1['UnitPrice'].to_numpy() * np.array([0.90])

In [30]:
df1.head()

,TransactionID,Date,CustomerID,ProductID,Category,Region,Quantity,UnitPrice,DiscountPct,NetSales,PaymentMethod,CustomerRating,Returned,DeliveryDays,GrossAmount,Sqrt_NetSales,NetSales_Normalized,Reshaped_Feature,DiscountedPrice
0,T00001,2024-11-23,1062,214,Grocery,North,2,17.38,10.0,31.28,Card,5.0,No,1.0,34.76,5.592853,0.001699,2,15.642
1,T00002,2024-01-16,1154,201,Sports,South,2,113.23,20.0,181.17,Wallet,2.0,No,5.0,226.46,13.459941,0.040581,2,101.907
2,T00003,2024-01-04,1071,209,Grocery,East,3,350.58,10.0,946.57,Card,1.0,No,2.0,1051.74,30.766378,0.239127,3,315.522
3,T00004,2024-07-02,1186,229,Sports,North,6,303.84,0.0,1823.04,Wallet,1.0,No,7.0,1823.04,42.697072,0.466484,6,273.456
4,T00005,2024-10-22,1169,214,Clothing,North,4,353.76,0.0,1415.04,UPI,1.0,No,5.0,1415.04,37.617017,0.360648,4,318.384


In [31]:
# Subtract row mean
num_data = df1[['Quantity', 'UnitPrice', 'NetSales']].dropna()
mean_centered = num_data.values - num_data.values.mean(axis=1, keepdims=True)

### 7. Linear Algebra with NumPy

In [32]:
# Solve system of equations, store solution in a DataFrame
A7 = np.array([[3, 1], [1, 2]])
b7 = np.array([9, 8])
solution7 = np.linalg.solve(A7, b7)
solution_df7 = pd.DataFrame({"variable": ["x", "y"], "value": solution7})
print("\nSolved system 3x+y=9, x+2y=8:\n", solution_df7)


Solved system 3x+y=9, x+2y=8:
   variable  value
0        x    2.0
1        y    3.0


In [33]:
# Dot product
dot_product = np.dot(df1['Quantity'].fillna(0), df1['UnitPrice'].fillna(0))

In [34]:
# Matrix multiplication on two DataFrames treated as matrices
matrix_A = df1[['Quantity', 'UnitPrice']].iloc[:5]
matrix_B = pd.DataFrame(np.random.rand(2, 3), index=['Quantity', 'UnitPrice'], columns=['Factor_1', 'Factor_2', 'Factor_3'])

# Perform matrix multiplication and store in a new DataFrame
df_matrix_mult = pd.DataFrame(matrix_A.values @ matrix_B.values, columns=matrix_B.columns)
df_matrix_mult

,Factor_1,Factor_2,Factor_3
0,18.858297,9.289590,8.366354
1,111.855175,55.535446,50.441691
2,343.137907,170.504470,154.999895
3,300.782655,149.309374,135.587974
4,347.221069,172.490792,156.764344


### 8. Handling Missing Data


In [35]:
# Interpolate missing values
df1['NetSales'] = df1['NetSales'].interpolate(method='linear')

In [36]:
# Replace missing values in discount with integer 0
df1['DiscountPct'] = df1['DiscountPct'].fillna(0)

In [37]:
# Replace outliers with median using mask
q25, q75 = df1['NetSales'].quantile(0.25), df1['NetSales'].quantile(0.75)
iqr = q75 - q25
outlier_mask = (df1['NetSales'] < (q25 - 1.5 * iqr)) | (df1['NetSales'] > (q75 + 1.5 * iqr))
df1.loc[outlier_mask, 'NetSales'] = df1['NetSales'].median()

### 9. Advanced Data Analysis

In [38]:
# Combination of groupby() and numpy
multi_grp = df1.groupby(['Region', 'Category'])['NetSales'].agg([np.mean, np.sum])

/tmp/ipykernel_8988/892874299.py:2: FutureWarning: The provided callable <function mean at 0x7a17f458e3e0> is currently using SeriesGroupBy.mean. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "mean" instead.
  multi_grp = df1.groupby(['Region', 'Category'])['NetSales'].agg([np.mean, np.sum])
/tmp/ipykernel_8988/892874299.py:2: FutureWarning: The provided callable <function sum at 0x7a17f458d300> is currently using SeriesGroupBy.sum. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "sum" instead.
  multi_grp = df1.groupby(['Region', 'Category'])['NetSales'].agg([np.mean, np.sum])


In [39]:
#Correlation matrix
corr_matrix = df1[['Quantity', 'UnitPrice', 'DiscountPct', 'NetSales']].corr()
corr_matrix

,Quantity,UnitPrice,DiscountPct,NetSales
Quantity,1.000000,-0.004275,-0.012522,0.458216
UnitPrice,-0.004275,1.000000,0.008910,0.564924
DiscountPct,-0.012522,0.008910,1.000000,-0.062266
NetSales,0.458216,0.564924,-0.062266,1.000000


### 10. DataFrame and Array Manipulation

In [40]:
#convert into numpy array
q_arr = df1['Quantity'].fillna(0).to_numpy()
u_arr = df1['UnitPrice'].fillna(0).to_numpy()

#calculated gross sales
gross_sales_arr = q_arr * u_arr

#added back to dataframe
df1['Calculated_GrossSales'] = gross_sales_arr

In [41]:
# create a random mask of boolean
random_mask = np.random.choice([True, False], size=len(df1), p=[0.5, 0.5])

# Filter DataFrame using the random mask
df_random_sample = df1[random_mask]

In [42]:
# Custom function to create 10% discount on order qty greater than 2
def custom_discount(qty, price):
    if qty > 2:
        return qty * price * 0.90
    return qty * price

# Vectorize the custom function for NumPy execution
vectorized_discount_fn = np.vectorize(custom_discount)

# Apply custom function across 'Quantity' and 'UnitPrice' columns
df1['Custom_Discounted_Revenue'] = vectorized_discount_fn(
    df1['Quantity'].fillna(0),
    df1['UnitPrice'].fillna(0)
)

### 11. Data Reshaping and Analysis

In [43]:
#1D array of size 12
flat_array = np.array([10, 20, 30, 40, 50, 60, 70, 80, 90, 100, 110, 120])

# Reshape 1D array to a 2D array of shape (4, 3)
reshaped_array = flat_array.reshape(4, 3)

# Convert reshaped array to a DataFrame
df_reshaped = pd.DataFrame(reshaped_array, columns=['Feature_A', 'Feature_B', 'Feature_C'])

In [44]:
# Satck two dataframe vertically
df_stacked = pd.concat([df1.iloc[:50], df1.iloc[50:100]], axis=0)

### 12. Time Series Analysis

In [45]:
# Convert to time series using pandas
df1['Date'] = pd.to_datetime(df1['Date'])
df1['Year'] = df1['Date'].dt.year
df1['Month'] = df1['Date'].dt.month
df1['DayOfWeek'] = df1['Date'].dt.day_name()

In [46]:
# Calculate Average of time series
df1 = df1.sort_values('Date')
df1['Rolling_Avg_Sales'] = df1['NetSales'].rolling(window=7, min_periods=1).mean()

In [47]:
# Calculate days elapsed since the earliest transaction date
start_date = df1['Date'].min()
df1['Days_Since_Start'] = (df1['Date'] - start_date).dt.days

# Calculate time difference between consecutive transaction rows in days
df1['Days_Between_Transactions'] = df1['Date'].diff().dt.days

In [48]:
df1

,TransactionID,Date,CustomerID,ProductID,Category,Region,Quantity,UnitPrice,DiscountPct,NetSales,...,Reshaped_Feature,DiscountedPrice,Calculated_GrossSales,Custom_Discounted_Revenue,Year,Month,DayOfWeek,Rolling_Avg_Sales,Days_Since_Start,Days_Between_Transactions
725,T00726,2024-01-02,1013,254,Clothing,West,7,322.28,0.0,1892.505000,...,7,290.052,2255.96,2030.364,2024,1,Tuesday,1892.505000,0,NaN
963,T00964,2024-01-02,1159,230,Clothing,South,2,96.61,0.0,1803.423333,...,2,86.949,193.22,193.220,2024,1,Tuesday,1847.964167,0,0.0
332,T00333,2024-01-03,1095,260,Electronics,East,1,269.22,0.0,849.480000,...,1,242.298,269.22,269.220,2024,1,Wednesday,1515.136111,1,1.0
365,T00366,2024-01-03,1010,209,Clothing,North,6,235.83,5.0,1344.230000,...,6,212.247,1414.98,1273.482,2024,1,Wednesday,1472.409583,1,0.0
727,T00728,2024-01-03,1146,251,Electronics,South,7,101.33,0.0,709.310000,...,7,91.197,709.31,638.379,2024,1,Wednesday,1319.789667,1,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
656,T00657,2024-12-29,1184,255,Grocery,North,8,231.59,0.0,1852.720000,...,8,208.431,1852.72,1667.448,2024,12,Sunday,1595.982857,362,1.0
321,T00322,2024-12-30,1089,229,Clothing,South,5,214.78,15.0,912.820000,...,5,193.302,1073.90,966.510,2024,12,Monday,1381.801429,363,1.0
698,T00699,2024-12-30,1131,255,Sports,North,8,69.05,0.0,2471.890000,...,8,62.145,552.40,497.160,2024,12,Monday,1520.620000,363,0.0
780,T00781,2024-12-30,1153,220,Home,South,3,129.71,0.0,389.130000,...,3,116.739,389.13,350.217,2024,12,Monday,1401.805714,363,0.0
